In [0]:
from pyspark.sql.functions import *

iot_df = spark.read.table("catalog_smartfactory.silver.streaming_iot_telemetry")

display(iot_df.limit(10))

timestamp,machine_id,temperature,vibration,pressure,rpm,power_consumption,failure_risk_score,event_timestamp,ingestion_timestamp,temperature_alerts,vibration_alerts,pressure_alerts,health_score,machine_status,event_date
2026-08-09T11:48:52.824Z,MCH-1011,103.69561520556262,4.133271723260657,108.521686860259,4276.260803647834,null,0.6570760898626467,2026-08-09T11:48:52.824Z,2026-08-09T11:52:59.014Z,1,1,0,30.29,CRITICAL,2026-08-11
2026-08-09T11:48:52.825Z,MCH-1027,72.46197629378001,0.7702474273150259,116.98312837334942,3573.3000505483897,null,0.1700455909231223,2026-08-09T11:48:52.825Z,2026-08-09T11:52:59.014Z,0,0,0,94.9,HEALTHY,2026-08-11
2026-08-09T11:48:52.825Z,MCH-1037,79.24678983996337,6.676756803295279,126.52077782705264,3100.5450513099404,null,0.41167312987997734,2026-08-09T11:48:52.825Z,2026-08-09T11:52:59.014Z,0,1,0,57.65,CRITICAL,2026-08-11
2026-08-09T11:48:52.826Z,MCH-1049,95.39054727556612,3.903215988420502,139.24841670435924,2476.5082543451776,null,0.4204508818692334,2026-08-09T11:48:52.826Z,2026-08-09T11:52:59.014Z,0,0,1,67.39,CRITICAL,2026-08-11
2026-08-09T11:48:52.826Z,MCH-1012,94.3773824987248,4.4190153219506545,131.26518621388277,3567.2911888969093,null,0.6461398958207755,2026-08-09T11:48:52.826Z,2026-08-09T11:52:59.014Z,0,1,1,30.62,CRITICAL,2026-08-11
2026-08-09T11:48:52.827Z,MCH-1023,88.31666583959904,5.3106803408022865,119.4297182643611,2523.672553862705,null,0.4493938392835787,2026-08-09T11:48:52.827Z,2026-08-09T11:52:59.014Z,0,1,0,56.52,CRITICAL,2026-08-11
2026-08-09T11:48:52.828Z,MCH-1014,74.77522080935347,1.2807610880779008,100.79385551009774,3737.657712451433,null,0.0075340167440672845,2026-08-09T11:48:52.828Z,2026-08-09T11:52:59.014Z,0,0,0,99.77,HEALTHY,2026-08-11
2026-08-09T11:48:52.828Z,MCH-1035,65.47687763143935,6.4737839476938515,136.15573491973888,1508.5224487949663,null,0.5553540994221844,2026-08-09T11:48:52.828Z,2026-08-09T11:52:59.014Z,0,1,1,33.34,CRITICAL,2026-08-11
2026-08-09T11:48:52.828Z,MCH-1014,90.43027337317329,4.359510405214978,100.51957894394786,2680.211374841081,null,0.5179677688157549,2026-08-09T11:48:52.828Z,2026-08-09T11:52:59.014Z,0,1,0,54.46,CRITICAL,2026-08-11
2026-08-09T11:48:52.829Z,MCH-1002,60.07706185078193,1.4493329061709943,85.78068225739199,4024.5310027644755,null,0.13872097211185686,2026-08-09T11:48:52.829Z,2026-08-09T11:52:59.014Z,0,0,0,95.84,HEALTHY,2026-08-11


In [0]:
ml_df = (
    iot_df
    .filter(col("temperature").isNotNull())
    .filter(col("vibration").isNotNull())
    .filter(col("pressure").isNotNull())
    .filter(col("rpm").isNotNull())
    .withColumn(
        "failure_label",
        when(
            (col("failure_risk_score") >= 0.7) |
            (col("temperature") > 105) |
            (col("vibration") > 4.5),
            1
        ).otherwise(0)
    )
    .select(
        "machine_id",
        "temperature",
        "vibration",
        "pressure",
        "rpm",
        "power_consumption",
        "health_score",
        "failure_risk_score",
        "failure_label"
    )
)

display(ml_df.limit(10))

machine_id,temperature,vibration,pressure,rpm,power_consumption,health_score,failure_risk_score,failure_label
MCH-1011,103.69561520556262,4.133271723260657,108.521686860259,4276.260803647834,null,30.29,0.6570760898626467,0
MCH-1027,72.46197629378001,0.7702474273150259,116.98312837334942,3573.3000505483897,null,94.9,0.1700455909231223,0
MCH-1037,79.24678983996337,6.676756803295279,126.52077782705264,3100.5450513099404,null,57.65,0.41167312987997734,1
MCH-1049,95.39054727556612,3.903215988420502,139.24841670435924,2476.5082543451776,null,67.39,0.4204508818692334,0
MCH-1012,94.3773824987248,4.4190153219506545,131.26518621388277,3567.2911888969093,null,30.62,0.6461398958207755,0
MCH-1023,88.31666583959904,5.3106803408022865,119.4297182643611,2523.672553862705,null,56.52,0.4493938392835787,1
MCH-1014,74.77522080935347,1.2807610880779008,100.79385551009774,3737.657712451433,null,99.77,0.0075340167440672845,0
MCH-1035,65.47687763143935,6.4737839476938515,136.15573491973888,1508.5224487949663,null,33.34,0.5553540994221844,1
MCH-1014,90.43027337317329,4.359510405214978,100.51957894394786,2680.211374841081,null,54.46,0.5179677688157549,0
MCH-1002,60.07706185078193,1.4493329061709943,85.78068225739199,4024.5310027644755,null,95.84,0.13872097211185686,0


In [0]:
pdf = ml_df.toPandas()

features = [
    "temperature",
    "vibration",
    "pressure",
    "rpm",
    "health_score"
]

X = pdf[features]
y = pdf["failure_label"]

In [0]:
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [0]:
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Workspace/Users/oussamaelal911@outlook.com/experiments/failure_prediction")

2026/08/11 14:24:05 INFO mlflow.tracking.fluent: Experiment with name '/Workspace/Users/oussamaelal911@outlook.com/experiments/failure_prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/4278251722955498', creation_time=1786458245914, experiment_id='4278251722955498', last_update_time=1786458245914, lifecycle_stage='active', name='/Users/oussamaelal911@outlook.com/experiments/failure_prediction', tags={'mlflow.experiment.sourceName': '/Users/oussamaelal911@outlook.com/experiments/failure_prediction',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'oussamaelal911@outlook.com',
 'mlflow.ownerId': '142474375480431'}>

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=6,
        random_state=42,
        class_weight="balanced"
    ))
])

with mlflow.start_run(run_name="factorypulse_failure_prediction_rf") as run:
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba)
    }

    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "n_estimators": 100,
        "max_depth": 6,
        "features": ",".join(features)
    })

    mlflow.log_metrics(metrics)

    signature = mlflow.models.infer_signature(X_train, pipeline.predict(X_train))

    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        registered_model_name="catalog_smartfactory.ml.failure_prediction_model",
        signature=signature,
        input_example=input_example
    )

    run_id = run.info.run_id

metrics


2026/08/11 14:31:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://adb-7405610420359805.5.azuredatabricks.net/ml/experiments/4278251722955498/models/m-7dc2b2191a8244d7bcfe0077fb04c2c8?o=7405610420359805
Successfully registered model 'catalog_smartfactory.ml.failure_prediction_model'.


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '1' of model 'catalog_smartfactory.ml.failure_prediction_model': https://adb-7405610420359805.5.azuredatabricks.net/explore/data/models/catalog_smartfactory/ml/failure_prediction_model/version/1?o=7405610420359805


{'accuracy': 0.972972972972973,
 'precision': 0.8823529411764706,
 'recall': 1.0,
 'f1': 0.9375,
 'roc_auc': np.float64(1.0)}

In [0]:
import mlflow.pyfunc

model_uri = "models:/catalog_smartfactory.ml.failure_prediction_model/1"
loaded_model = mlflow.pyfunc.load_model(model_uri)


In [0]:
import pandas as pd
latest_pdf = (
    ml_df
    .limit(10000)
    .toPandas()
)

prediction_features = latest_pdf[features]

latest_pdf["predicted_failure"] = loaded_model.predict(prediction_features)
latest_pdf["prediction_timestamp"] = pd.Timestamp.utcnow()
latest_pdf["failure_probability"] = loaded_model.predict_proba(prediction_features)[:, 1]


In [0]:
import pandas as pd
latest_pdf = (
    ml_df
    .limit(10000)
    .toPandas()
)

prediction_features = latest_pdf[features]

latest_pdf["predicted_failure"] = loaded_model.predict(prediction_features)
latest_pdf["prediction_timestamp"] = pd.Timestamp.utcnow()

In [0]:
predictions_sdf = spark.createDataFrame(latest_pdf)

predictions_sdf.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("catalog_smartfactory.ml.machine_failure_predictions")